In [1]:
import pandas as pd
import numpy as np
import os

print("="*60)
print("STEP 1: DOWNLOAD & LOAD MOVIELENS 1M")
print("="*60)

STEP 1: DOWNLOAD & LOAD MOVIELENS 1M


In [2]:
# Download MovieLens 1M
!wget -q https://files.grouplens.org/datasets/movielens/ml-1m.zip
!unzip -q -y ml-1m.zip

UnZip 6.00 of 20 April 2009, by Debian. Original by Info-ZIP.

Usage: unzip [-Z] [-opts[modifiers]] file[.zip] [list] [-x xlist] [-d exdir]
  Default action is to extract files in list, except those in xlist, to exdir;
  file[.zip] may be a wildcard.  -Z => ZipInfo mode ("unzip -Z" for usage).

  -p  extract files to pipe, no messages     -l  list files (short format)
  -f  freshen existing files, create none    -t  test compressed archive data
  -u  update files, create if necessary      -z  display archive comment only
  -v  list verbosely/show version info       -T  timestamp archive to latest
  -x  exclude files that follow (in xlist)   -d  extract files into exdir
modifiers:
  -n  never overwrite existing files         -q  quiet mode (-qq => quieter)
  -o  overwrite files WITHOUT prompting      -a  auto-convert any text files
  -j  junk paths (do not make directories)   -aa treat ALL files as text
  -U  use escapes for all non-ASCII Unicode  -UU ignore any Unicode fields
  -C  mat

In [3]:
# Load raw data
ratings = pd.read_csv(
    "ml-1m/ratings.dat",
    sep="::",
    engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"],
    dtype={
        "user_id": np.int32,
        "movie_id": np.int32,
        "rating": np.float32,
        "timestamp": np.int32,
    },
)

print(f"✓ Loaded {len(ratings)} ratings")

✓ Loaded 1000209 ratings


In [4]:
print("\n" + "="*60)
print("STEP 2: PREPROCESS DATA")
print("="*60)

# Filter high ratings (>=4) to indicate interest
ratings = ratings[ratings["rating"] >= 4.0]
print(f"After filtering rating>=4: {len(ratings)} interactions")

# Sort by user and timestamp
ratings = ratings.sort_values(["user_id", "timestamp"])

# Rename columns to GRU4Rec format
ratings = ratings.rename(
    columns={
        "user_id": "SessionId",
        "movie_id": "ItemId",
        "timestamp": "Time",
    }
)
data = ratings[["SessionId", "ItemId", "Time"]]

# Remove sessions with only 1 interaction
session_lengths = data.groupby("SessionId").size()
valid_sessions = session_lengths[session_lengths >= 2].index
data = data[data["SessionId"].isin(valid_sessions)]
print(
    f"After removing single-item sessions: "
    f"{len(data)} interactions, {data['SessionId'].nunique()} sessions"
)



STEP 2: PREPROCESS DATA
After filtering rating>=4: 575281 interactions
After removing single-item sessions: 575280 interactions, 6037 sessions


In [5]:
def train_test_split(data, min_test_length=2):
    train_list = []
    test_list = []
    
    for session_id in data["SessionId"].unique():
        session_data = data[data["SessionId"] == session_id].copy()
        session_len = len(session_data)
        
        # Only split if session has at least 3 items (1 for train, 2+ for test)
        if session_len >= 3:
            # Put last 2 items in test, rest in train
            split_point = -2
            train_list.append(session_data.iloc[:split_point])
            test_list.append(session_data.iloc[split_point:])
        elif session_len == 2:
            # If only 2 items, put both in test (minimum viable test session)
            test_list.append(session_data)
        # Sessions with 1 item are already filtered out
    
    train_df = pd.concat(train_list) if train_list else pd.DataFrame()
    test_df = pd.concat(test_list) if test_list else pd.DataFrame()
    return train_df, test_df


In [6]:
train_full, test = train_test_split(data)

# Save to TSV files
train_full.to_csv("movielens1m_train_full.tsv", sep="\t", index=False)
test.to_csv("movielens1m_test.tsv", sep="\t", index=False)

print(
    f"✓ Train: {len(train_full)} interactions, "
    f"{train_full['SessionId'].nunique()} sessions"
)
print(
    f"✓ Test: {len(test)} interactions, "
    f"{test['SessionId'].nunique()} sessions"
)
print(
    f"  Test avg session length: "
    f"{len(test) / test['SessionId'].nunique():.2f}"
)


✓ Train: 563206 interactions, 6035 sessions
✓ Test: 12074 interactions, 6037 sessions
  Test avg session length: 2.00


In [7]:
print("\n" + "="*60)
print("STEP 3: CLONE & FIX GRU4REC")
print("="*60)

!git clone https://github.com/hidasib/GRU4Rec_PyTorch_Official.git

os.chdir("GRU4Rec_PyTorch_Official")

# Copy data files
!cp ../movielens1m_train_full.tsv .
!cp ../movielens1m_test.tsv .



STEP 3: CLONE & FIX GRU4REC
Cloning into 'GRU4Rec_PyTorch_Official'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 75 (delta 20), reused 15 (delta 15), pack-reused 48 (from 1)
Receiving objects: 100% (75/75), 362.75 KiB | 1.66 MiB/s, done.
Resolving deltas: 100% (35/35), done.


In [8]:
fix_script = r"""
import re

with open('gru4rec_pytorch.py', 'r') as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    # Fix line 186 specifically - the problematic one
    if 'self.Wy.weight.set_(torch.tensor(self._init_numpy_weights((self.n_items, self.layers[-1]))' in line:
        # Replace with proper dtype specification
        line = line.replace(
            'torch.tensor(self._init_numpy_weights((self.n_items, self.layers[-1])), device=',
            'torch.tensor(self._init_numpy_weights((self.n_items, self.layers[-1])), dtype=torch.float32, device='
        )
    
    # Fix line 188 - similar issue
    if 'self.Wrz.weight.set_(torch.tensor(self._init_numpy_weights((self.layers[0], 3*self.layers[0]))' in line:
        line = line.replace(
            'torch.tensor(self._init_numpy_weights((self.layers[0], 3*self.layers[0])), device=',
            'torch.tensor(self._init_numpy_weights((self.layers[0], 3*self.layers[0])), dtype=torch.float32, device='
        )
    
    # Fix line 190 - similar issue
    if 'self.Wh.weight.set_(torch.tensor(self._init_numpy_weights((self.layers[0], self.layers[0]))' in line:
        line = line.replace(
            'torch.tensor(self._init_numpy_weights((self.layers[0], self.layers[0])), device=',
            'torch.tensor(self._init_numpy_weights((self.layers[0], self.layers[0])), dtype=torch.float32, device='
        )
    
    # Fix any other torch.tensor calls with numpy arrays
    if 'torch.tensor(np.vstack(m), device=' in line:
        line = line.replace(
            'torch.tensor(np.vstack(m), device=',
            'torch.tensor(np.vstack(m), dtype=torch.float32, device='
        )
    
    if 'torch.tensor(np.vstack(m2), device=' in line:
        line = line.replace(
            'torch.tensor(np.vstack(m2), device=',
            'torch.tensor(np.vstack(m2), dtype=torch.float32, device='
        )
    
    new_lines.append(line)

with open('gru4rec_pytorch.py', 'w') as f:
    f.writelines(new_lines)

print('Fixed all dtype issues')
"""

with open("fix.py", "w") as f:
    f.write(fix_script)

!python fix.py
print("✓ Fixed compatibility issues")


Fixed all dtype issues
✓ Fixed compatibility issues


In [9]:
print("\n" + "="*60)
print("STEP 4: TRAIN GRU4REC")
print("="*60)

!python run.py movielens1m_train_full.tsv \
    -t movielens1m_test.tsv \
    -m 5 10 20 \
    -ps "loss=bpr-max,constrained_embedding=True,embedding=0,elu_param=0.5,layers=100,batch_size=512,dropout_p_embed=0.0,dropout_p_hidden=0.5,learning_rate=0.05,momentum=0.2,n_sample=2048,sample_alpha=0.75,bpreg=1.9,logq=0.0,n_epochs=10" \
    -d cuda \
    -s gru4rec_movielens1m.pt

print("\n" + "="*60)
print("✓ ALL DONE!")
print("="*60)


STEP 4: TRAIN GRU4REC
Creating GRU4Rec model on device "cuda"
SET   loss                    TO   bpr-max   (type: <class 'str'>)
SET   constrained_embedding   TO   True      (type: <class 'bool'>)
SET   embedding               TO   0         (type: <class 'int'>)
SET   elu_param               TO   0.5       (type: <class 'float'>)
SET   layers                  TO   [100]     (type: <class 'list'>)
SET   batch_size              TO   512       (type: <class 'int'>)
SET   dropout_p_embed         TO   0.0       (type: <class 'float'>)
SET   dropout_p_hidden        TO   0.5       (type: <class 'float'>)
SET   learning_rate           TO   0.05      (type: <class 'float'>)
SET   momentum                TO   0.2       (type: <class 'float'>)
SET   n_sample                TO   2048      (type: <class 'int'>)
SET   sample_alpha            TO   0.75      (type: <class 'float'>)
SET   bpreg                   TO   1.9       (type: <class 'float'>)
SET   logq                    TO   0.0       (type